# PlantVillage Teacher Reproduction (39 classes)

**This notebook is deliberately split into preflight and training.** Run the preflight first and review its output. The 1,000-epoch training cell requires you to type `TRAIN` before it starts.

The paper's reported counts do not match the 55,448-row public 39-class dataset exactly, and this source does not provide physical-leaf IDs. See the project README. Do not describe the generated row-level split as the paper's exact split or as leaf-grouped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, torch, torchvision
PROJECT_DIR = Path('/content/drive/MyDrive/plant_disease_vit/teacher_reproduction')
SRC_DIR = PROJECT_DIR / 'src'
SPLIT_PATH = PROJECT_DIR / 'splits' / 'plant_village_39class_seed42.json'
CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints'
RESULTS_DIR = PROJECT_DIR / 'results'
CACHE_DIR = Path('/content/hf_cache')
for p in [SRC_DIR, PROJECT_DIR / 'splits', CHECKPOINT_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
if not (SRC_DIR / 'train.py').exists():
    raise FileNotFoundError(f'Expected source files under {SRC_DIR}. Upload the extracted teacher_reproduction folder to MyDrive/plant_disease_vit/teacher_reproduction first.')
sys.path.insert(0, str(SRC_DIR))
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
else: raise RuntimeError('No GPU is assigned. In Colab choose Runtime > Change runtime type > GPU, then rerun this cell.')

In [ ]:
%pip -q install 'datasets>=3.3.2,<4.0' 'scikit-learn>=1.5' 'tqdm>=4.66'
print('Dependencies ready. If Colab asks for a runtime restart, restart and rerun the setup cell.')

## Preflight: inspect dataset and create a fixed manifest (no training)
This uses every row in the 39-class dataset and creates one reproducible stratified 80/10/10 split. The actual counts will be printed; the script will not duplicate examples to force the inconsistent counts printed in the paper.

In [ ]:
from data import load_hf_dataset, prepare_or_load_splits
DATASET_ID = 'Project-AgML/plant_village_classification'
SEED = 42
ds, class_names = load_hf_dataset(DATASET_ID, cache_dir=str(CACHE_DIR))
manifest = prepare_or_load_splits(ds, class_names, str(SPLIT_PATH), seed=SEED, dataset_id=DATASET_ID)
print('Dataset:', DATASET_ID)
print('Rows:', len(ds), '| classes:', len(class_names))
print('First/last classes:', class_names[:3], '...', class_names[-3:])
print('Split counts:', manifest['counts'])
import numpy as np
print('Per-class counts (train / validation / test):')
for class_name in class_names:
    counts = [manifest['class_counts'][s][class_name] for s in ['train','validation','test']]
    print(f'{class_name}: {counts[0]} / {counts[1]} / {counts[2]}')
print('Split method:', manifest['split_method'])
print('Manifest:', SPLIT_PATH)
print('Image sample size:', ds[0]['image'].size, '| label:', class_names[int(ds[0]['label'])])
print('Limitations to keep in the paper:')
for item in manifest['limitations']: print('-', item)
assert len(class_names) == 39 and set(manifest['splits']) == {'train','validation','test'}
print('PRE-FLIGHT PASSED: split rows are disjoint and cover all source rows. This does not prove physical-leaf separation.')

## Preflight: GPU memory/compute benchmark (no full training)
This performs a single real forward/backward pass using the selected micro-batch. It does not save model weights. If it fails with CUDA OOM, change `MICRO_BATCH_SIZE` to 8 and `GRAD_ACCUM_STEPS` to 8, then rerun.

In [ ]:
from model import build_teacher, count_parameters
from torch import nn
MICRO_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
model = build_teacher(num_classes=len(class_names)).cuda().train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=True)
criterion = nn.CrossEntropyLoss()
dummy_x = torch.randn(MICRO_BATCH_SIZE, 3, 224, 224, device='cuda')
dummy_y = torch.randint(0, len(class_names), (MICRO_BATCH_SIZE,), device='cuda')
torch.cuda.reset_peak_memory_stats()
optimizer.zero_grad(set_to_none=True)
with torch.autocast(device_type='cuda', dtype=torch.float16):
    loss = criterion(model(dummy_x), dummy_y)
scaler.scale(loss).backward()
scaler.step(optimizer); scaler.update()
torch.cuda.synchronize()
print('Parameters:', f'{count_parameters(model):,}')
print('Micro-batch:', MICRO_BATCH_SIZE, '| accumulation:', GRAD_ACCUM_STEPS, '| effective batch:', MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS)
print('Peak allocated GiB:', round(torch.cuda.max_memory_allocated()/1024**3, 2))
del model, optimizer, scaler, dummy_x, dummy_y, loss
torch.cuda.empty_cache()
print('GPU preflight passed. This is a memory check, not a training-speed estimate.')

## Train the teacher (1,000 epochs; resumable)
**Before running:** verify the dataset counts and split limitations printed above. Check that the `teacher_reproduction` folder is in Drive and the existing baseline checkpoint is not inside this output directory. The code saves `last.pt` every five epochs and automatically resumes it on the next run.

In [ ]:
confirm = input('Review preflight results. Type TRAIN to begin/resume the 1000-epoch run: ').strip()
if confirm != 'TRAIN':
    print('Training not started.')
else:
    import subprocess
    cmd = [sys.executable, str(SRC_DIR / 'train.py'),
           '--dataset_id', DATASET_ID,
           '--cache_dir', str(CACHE_DIR),
           '--split_manifest', str(SPLIT_PATH),
           '--output_dir', str(CHECKPOINT_DIR),
           '--results_dir', str(RESULTS_DIR),
           '--epochs', '1000',
           '--micro_batch_size', str(MICRO_BATCH_SIZE),
           '--grad_accum_steps', str(GRAD_ACCUM_STEPS),
           '--eval_batch_size', '32',
           '--num_workers', '2',
           '--seed', str(SEED),
           '--save_every', '5',
           '--resume', 'auto']
    print('Launching:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

## Evaluate the held-out test split (only after training is complete)
Run after `last.pt` shows epoch 1000 and `best_model.pt` exists. If the runtime has restarted, rerun the setup and preflight cells first; the saved manifest ensures evaluation uses the same split.

In [ ]:
checkpoint = CHECKPOINT_DIR / 'best_model.pt'
if not checkpoint.exists(): raise FileNotFoundError(f'Missing {checkpoint}; finish training first.')
subprocess.run([sys.executable, str(SRC_DIR / 'evaluate.py'),
                '--dataset_id', DATASET_ID, '--cache_dir', str(CACHE_DIR),
                '--split_manifest', str(SPLIT_PATH), '--checkpoint', str(checkpoint),
                '--output_dir', str(RESULTS_DIR), '--eval_batch_size', '32',
                '--num_workers', '2', '--seed', str(SEED)], check=True)
print('Evaluation JSON:', RESULTS_DIR / 'test_metrics.json')